# 05 V2 — Training Data Preparation

Converts Gemini-generated schema into training format for Qwen3-VL fine-tuning.

**Input**: `data/generated/*.json` (Gemini outputs with validation metadata)
**Output**: `data/processed/train.jsonl` + `eval.jsonl` (90/10 split)

Each training example is a conversation:
```json
{
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": [{"type": "image", "image": "path"}, {"type": "text", "text": "HTML..."}]},
    {"role": "assistant", "content": "{JSON-LD}"}
  ]
}
```

In [ ]:
import json, glob, hashlib, random, os
from pathlib import Path
from collections import Counter

PROJECT = Path('..').resolve()
GEN_DIR = PROJECT / 'data' / 'generated'
SHOT_DIR = PROJECT / 'data' / 'screenshots_v2'
PAGES_PATH = PROJECT / 'data' / 'processed' / 'pages_for_generation.jsonl'
OUT_DIR = PROJECT / 'data' / 'processed'

# Quality threshold — only include examples above this score
MIN_QUALITY = 0.3
TRAIN_SPLIT = 0.9
MAX_HTML_CHARS = 12_000  # must match what Gemini saw

SYSTEM_PROMPT = (
    'You are a schema.org JSON-LD expert. Given a screenshot and HTML of a web page, '
    'generate the optimal schema.org JSON-LD markup. Output ONLY valid JSON-LD with '
    'no markdown formatting, no explanation, and no code fences. '
    'Use the most specific @type available. Include all extractable properties. '
    'Always include "@context": "https://schema.org".'
)

print(f'Min quality: {MIN_QUALITY}')
print(f'Train split: {TRAIN_SPLIT}')

In [ ]:
# Load generated results
gen_files = glob.glob(str(GEN_DIR / '*.json'))
results = []
for f in gen_files:
    with open(f) as fh:
        r = json.load(fh)
        if r['valid'] and r['quality_score'] >= MIN_QUALITY:
            results.append(r)

print(f'Total generated files: {len(gen_files):,}')
print(f'Passed quality filter (>= {MIN_QUALITY}): {len(results):,}')

# Load source pages for HTML
pages_by_url = {}
with open(PAGES_PATH) as f:
    for line in f:
        rec = json.loads(line)
        if rec['url'] not in pages_by_url:
            pages_by_url[rec['url']] = rec

print(f'Source pages loaded: {len(pages_by_url):,}')

In [ ]:
# Build training examples
examples = []
skipped_no_html = 0
skipped_no_shot = 0

for r in results:
    url = r['url']
    h = hashlib.md5(url.encode()).hexdigest()
    shot_path = SHOT_DIR / f'{h}.png'
    
    # Get source HTML
    source = pages_by_url.get(url)
    if not source:
        skipped_no_html += 1
        continue
    
    if not shot_path.exists():
        skipped_no_shot += 1
        continue
    
    html = source['html'][:MAX_HTML_CHARS]
    if len(source['html']) > MAX_HTML_CHARS:
        html += '\n<!-- [HTML truncated] -->'
    
    # Clean up the JSON-LD output (re-format for consistency)
    try:
        parsed = json.loads(r['generated_jsonld'])
        jsonld_clean = json.dumps(parsed, indent=2, ensure_ascii=False)
    except:
        continue
    
    example = {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': str(shot_path)},
                {'type': 'text', 'text': html}
            ]},
            {'role': 'assistant', 'content': jsonld_clean}
        ],
        'metadata': {
            'url': url,
            'schema_types': r['schema_types'],
            'quality_score': r['quality_score'],
            'property_count': r['property_count'],
        }
    }
    examples.append(example)

print(f'Training examples built: {len(examples):,}')
print(f'Skipped (no HTML): {skipped_no_html}')
print(f'Skipped (no screenshot): {skipped_no_shot}')

In [ ]:
# Stats before split
types = Counter()
qualities = []
for ex in examples:
    m = ex['metadata']
    for t in m['schema_types']:
        types[t] += 1
    qualities.append(m['quality_score'])

print(f'Type distribution ({len(types)} unique):')
for t, n in types.most_common(20):
    print(f'  {t:35s} {n:5,}')

print(f'\nQuality: mean={sum(qualities)/len(qualities):.3f}, '
      f'median={sorted(qualities)[len(qualities)//2]:.3f}')

In [ ]:
# Stratified split by primary type
random.seed(42)

by_type = {}
for ex in examples:
    t = ex['metadata']['schema_types'][0] if ex['metadata']['schema_types'] else 'Unknown'
    by_type.setdefault(t, []).append(ex)

train_set = []
eval_set = []

for t, exs in by_type.items():
    random.shuffle(exs)
    split_idx = max(1, int(len(exs) * TRAIN_SPLIT))
    train_set.extend(exs[:split_idx])
    eval_set.extend(exs[split_idx:])

random.shuffle(train_set)
random.shuffle(eval_set)

print(f'Train: {len(train_set):,}')
print(f'Eval:  {len(eval_set):,}')
print(f'Total: {len(train_set) + len(eval_set):,}')

In [ ]:
# Save
train_path = OUT_DIR / 'train.jsonl'
eval_path = OUT_DIR / 'eval.jsonl'

for path, data in [(train_path, train_set), (eval_path, eval_set)]:
    with open(path, 'w') as f:
        for ex in data:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')

train_mb = train_path.stat().st_size / 1e6
eval_mb = eval_path.stat().st_size / 1e6

print(f'Saved:')
print(f'  {train_path} ({train_mb:.1f} MB, {len(train_set):,} examples)')
print(f'  {eval_path} ({eval_mb:.1f} MB, {len(eval_set):,} examples)')
print(f'\nNext: 06_v2 or configs/training_config.yaml for fine-tuning')